In [ ]:
# ================================================================
# 🍅 TOMATO — INTERNAL + EXTERNAL DATA PREPROCESSING
# ================================================================
#
# RAW DATA:
#   1st Raw_Data/Tomato
#
# OUTPUT:
#   2nd_Preprocessed_Data/Tomato
#
# PROCESS:
#   1. Read locked raw images
#   2. Verify image
#   3. Convert RGB
#   4. Resize to 224 x 224
#   5. Save standardized PNG
#   6. Create preprocessing report
#   7. Final verification
#
# IMPORTANT:
#   ❌ Raw data is NOT modified
#   ❌ Raw data is NOT deleted
#   ❌ Raw data is NOT renamed
#   ✅ New processed dataset is created
# ================================================================

import os
import shutil
import hashlib
import csv

from PIL import Image
from google.colab import drive

# ================================================================
# 1. GOOGLE DRIVE
# ================================================================

drive.mount("/content/drive")

# ================================================================
# 2. PATHS
# ================================================================

PROJECT = (
    "/content/drive/MyDrive/"
    "Plant Disease Detection (Computer Vision)"
)

RAW_TOMATO = os.path.join(
    PROJECT,
    "1st_Raw_Data"
)

# Correct existing raw-data folder
RAW_TOMATO = os.path.join(
    PROJECT,
    "1st Raw_Data",
    "Tomato"
)

PROCESSED_ROOT = os.path.join(
    PROJECT,
    "2nd_Preprocessed_Data",
    "Tomato"
)

INTERNAL_RAW = os.path.join(
    RAW_TOMATO,
    "Internal_PlantVillage"
)

EXTERNAL_RAW = os.path.join(
    RAW_TOMATO,
    "External_Natural",
    "Verified_300"
)

INTERNAL_OUT = os.path.join(
    PROCESSED_ROOT,
    "Internal_PlantVillage"
)

EXTERNAL_OUT = os.path.join(
    PROCESSED_ROOT,
    "External_Natural"
)

CLASSES = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)

TARGET_SIZE = (224, 224)

# ================================================================
# 3. CHECK RAW DATA
# ================================================================

print("=" * 72)
print("🍅 TOMATO — DATA PREPROCESSING")
print("=" * 72)

print("\nChecking locked raw dataset...")

if not os.path.exists(RAW_TOMATO):
    raise Exception(
        "❌ Tomato raw-data folder not found."
    )

if not os.path.exists(INTERNAL_RAW):
    raise Exception(
        "❌ Internal PlantVillage folder not found."
    )

if not os.path.exists(EXTERNAL_RAW):
    raise Exception(
        "❌ External Verified_300 folder not found."
    )

print("✅ Raw Tomato dataset found")
print("🔒 Raw dataset will NOT be modified")

# ================================================================
# 4. CREATE PROCESSED FOLDERS
# ================================================================

if os.path.exists(PROCESSED_ROOT):

    print("\n⚠️ Existing processed Tomato folder found.")

    print("Removing OLD processed output only...")

    shutil.rmtree(PROCESSED_ROOT)

os.makedirs(PROCESSED_ROOT)

for base in [
    INTERNAL_OUT,
    EXTERNAL_OUT
]:

    for cls in CLASSES:

        os.makedirs(
            os.path.join(base, cls),
            exist_ok=True
        )

print("✅ Fresh preprocessing folders created")

# ================================================================
# 5. PREPROCESSING FUNCTION
# ================================================================

def preprocess_dataset(
    source_root,
    output_root,
    source_name
):

    report = []

    print("\n" + "=" * 72)
    print(f"PROCESSING: {source_name}")
    print("=" * 72)

    for cls in CLASSES:

        source_folder = os.path.join(
            source_root,
            cls
        )

        output_folder = os.path.join(
            output_root,
            cls
        )

        if not os.path.exists(source_folder):

            print(
                f"❌ {cls}: source folder missing"
            )

            continue

        files = [
            f for f in os.listdir(source_folder)
            if f.lower().endswith(
                IMAGE_EXTENSIONS
            )
        ]

        files.sort()

        print(
            f"\n{cls}"
        )

        print(
            f"Raw images found: {len(files)}"
        )

        processed = 0
        failed = 0

        output_hashes = set()

        for index, filename in enumerate(
            files,
            start=1
        ):

            source_path = os.path.join(
                source_folder,
                filename
            )

            output_filename = (
                f"{cls}_{index:03d}.png"
            )

            output_path = os.path.join(
                output_folder,
                output_filename
            )

            try:

                # ------------------------------------------------
                # OPEN + VERIFY
                # ------------------------------------------------

                with Image.open(source_path) as img:

                    img.verify()

                # Re-open after verify
                with Image.open(source_path) as img:

                    # ------------------------------------------------
                    # CONVERT TO RGB
                    # ------------------------------------------------

                    img = img.convert("RGB")

                    # ------------------------------------------------
                    # RESIZE
                    # ------------------------------------------------

                    img = img.resize(
                        TARGET_SIZE,
                        Image.Resampling.LANCZOS
                    )

                    # ------------------------------------------------
                    # SAVE STANDARDIZED IMAGE
                    # ------------------------------------------------

                    img.save(
                        output_path,
                        format="PNG",
                        optimize=True
                    )

                # ------------------------------------------------
                # VERIFY OUTPUT
                # ------------------------------------------------

                with Image.open(output_path) as check:

                    check.verify()

                    if check.size != TARGET_SIZE:
                        raise Exception(
                            "Incorrect output size"
                        )

                    if check.mode != "RGB":
                        raise Exception(
                            "Incorrect color mode"
                        )

                # ------------------------------------------------
                # HASH OUTPUT
                # ------------------------------------------------

                with open(
                    output_path,
                    "rb"
                ) as f:

                    output_hash = hashlib.sha256(
                        f.read()
                    ).hexdigest()

                if output_hash in output_hashes:

                    print(
                        f"⚠️ Duplicate output: "
                        f"{output_filename}"
                    )

                output_hashes.add(
                    output_hash
                )

                processed += 1

                report.append({

                    "Source": source_name,
                    "Class": cls,
                    "Original_File": filename,
                    "Processed_File": output_filename,
                    "Original_Format":
                        os.path.splitext(
                            filename
                        )[1].upper(),
                    "Processed_Format": "PNG",
                    "Processed_Size":
                        "224x224",
                    "Color_Mode": "RGB",
                    "Status": "SUCCESS"

                })

            except Exception as e:

                failed += 1

                report.append({

                    "Source": source_name,
                    "Class": cls,
                    "Original_File": filename,
                    "Processed_File": "",
                    "Original_Format":
                        os.path.splitext(
                            filename
                        )[1].upper(),
                    "Processed_Format": "",
                    "Processed_Size": "",
                    "Color_Mode": "",
                    "Status":
                        f"FAILED: {str(e)}"

                })

        print(
            f"Processed : {processed}"
        )

        print(
            f"Failed    : {failed}"
        )

    return report


# ================================================================
# 6. PROCESS INTERNAL
# ================================================================

internal_report = preprocess_dataset(
    INTERNAL_RAW,
    INTERNAL_OUT,
    "Internal_PlantVillage"
)

# ================================================================
# 7. PROCESS EXTERNAL
# ================================================================

external_report = preprocess_dataset(
    EXTERNAL_RAW,
    EXTERNAL_OUT,
    "External_Natural"
)

all_report = (
    internal_report +
    external_report
)

# ================================================================
# 8. SAVE REPORT
# ================================================================

REPORT_DIR = os.path.join(
    PROCESSED_ROOT,
    "Preprocessing_Report"
)

os.makedirs(
    REPORT_DIR,
    exist_ok=True
)

REPORT_FILE = os.path.join(
    REPORT_DIR,
    "Tomato_Preprocessing_Report.csv"
)

with open(
    REPORT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    fieldnames = [
        "Source",
        "Class",
        "Original_File",
        "Processed_File",
        "Original_Format",
        "Processed_Format",
        "Processed_Size",
        "Color_Mode",
        "Status"
    ]

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()

    writer.writerows(
        all_report
    )

print("\n" + "=" * 72)
print("📄 PREPROCESSING REPORT SAVED")
print("=" * 72)

print(REPORT_FILE)

# ================================================================
# 9. FINAL PROCESSED DATA VERIFICATION
# ================================================================

print("\n" + "=" * 72)
print("🍅 FINAL PREPROCESSED DATA VERIFICATION")
print("=" * 72)

final_ok = True

for source_name, base in [
    (
        "INTERNAL",
        INTERNAL_OUT
    ),
    (
        "EXTERNAL",
        EXTERNAL_OUT
    )
]:

    print("\n" + "-" * 72)
    print(source_name)
    print("-" * 72)

    for cls in CLASSES:

        folder = os.path.join(
            base,
            cls
        )

        files = [
            f for f in os.listdir(folder)
            if f.lower().endswith(".png")
        ]

        valid = 0
        invalid = 0
        hashes = set()
        duplicates = 0

        for file in files:

            path = os.path.join(
                folder,
                file
            )

            try:

                with Image.open(path) as img:

                    if (
                        img.size != TARGET_SIZE
                        or img.mode != "RGB"
                    ):

                        invalid += 1
                        continue

                    img.verify()

                with open(
                    path,
                    "rb"
                ) as f:

                    h = hashlib.sha256(
                        f.read()
                    ).hexdigest()

                if h in hashes:

                    duplicates += 1

                else:

                    hashes.add(h)

                valid += 1

            except:

                invalid += 1

        print(
            f"{cls:15s} | "
            f"Files: {len(files):3d} | "
            f"Valid: {valid:3d} | "
            f"Invalid: {invalid:2d} | "
            f"Duplicate: {duplicates:2d}"
        )

        # ------------------------------------------------
        # Acceptance
        # ------------------------------------------------

        # Internal Healthy = 299
        if (
            source_name == "INTERNAL"
            and cls == "Healthy"
        ):

            expected = 299

        else:

            expected = 300

        if (
            valid != expected
            or invalid != 0
            or duplicates != 0
        ):

            final_ok = False

# ================================================================
# 10. FINAL STATUS
# ================================================================

print("\n" + "=" * 72)

if final_ok:

    print("🎉 TOMATO PREPROCESSING COMPLETED SUCCESSFULLY")
    print("=" * 72)

    print("""
PREPROCESSING:

✅ Internal + External processed
✅ RGB conversion
✅ Resized to 224 × 224
✅ Saved as standardized PNG
✅ Output images verified
✅ No corrupt processed images
✅ No duplicate processed files
✅ Raw dataset untouched

OUTPUT:

2nd_Preprocessed_Data
└── Tomato
    ├── Internal_PlantVillage
    │   ├── Healthy
    │   ├── Early_Blight
    │   └── Late_Blight
    │
    ├── External_Natural
    │   ├── Healthy
    │   ├── Early_Blight
    │   └── Late_Blight
    │
    └── Preprocessing_Report

NEXT:
➡️ Train / Validation / Test Split
➡️ CNN Model Training
""")

else:

    print("❌ PREPROCESSING VERIFICATION FAILED")
    print("Do NOT start model training yet.")
    print("Check the preprocessing report.")

print("=" * 72)

Mounted at /content/drive
🍅 TOMATO — DATA PREPROCESSING

Checking locked raw dataset...
✅ Raw Tomato dataset found
🔒 Raw dataset will NOT be modified
✅ Fresh preprocessing folders created

PROCESSING: Internal_PlantVillage

Healthy
Raw images found: 300
⚠️ Duplicate output: Healthy_239.png
Processed : 300
Failed    : 0

Early_Blight
Raw images found: 300
Processed : 300
Failed    : 0

Late_Blight
Raw images found: 300
Processed : 300
Failed    : 0

PROCESSING: External_Natural

Healthy
Raw images found: 300
Processed : 300
Failed    : 0

Early_Blight
Raw images found: 300
Processed : 300
Failed    : 0

Late_Blight
Raw images found: 300
Processed : 300
Failed    : 0

📄 PREPROCESSING REPORT SAVED
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/2nd_Preprocessed_Data/Tomato/Preprocessing_Report/Tomato_Preprocessing_Report.csv

🍅 FINAL PREPROCESSED DATA VERIFICATION

------------------------------------------------------------------------
INTERNAL
--------------------------

In [ ]:
# ================================================================
# 🍅 TOMATO — FINAL PREPROCESSING LOCK
# INTERNAL + EXTERNAL
# ================================================================

import os
import hashlib
from PIL import Image
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

PREP = os.path.join(
    PROJECT,
    "2nd_Preprocessed_Data",
    "Tomato"
)

CLASSES = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

SOURCES = [
    "Internal_PlantVillage",
    "External_Natural"
]

EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")

print("=" * 72)
print("🍅 TOMATO — FINAL PREPROCESSING LOCK")
print("=" * 72)

print("""
READ-ONLY CHECK

✅ No raw data modification
✅ No deletion
✅ No movement
✅ No resizing
✅ No reprocessing
""")

final_pass = True
total_files = 0

for source in SOURCES:

    print("\n" + "=" * 72)
    print(source.upper())
    print("=" * 72)

    for cls in CLASSES:

        folder = os.path.join(
            PREP,
            source,
            cls
        )

        if not os.path.exists(folder):

            print(f"❌ {source} | {cls} | Folder missing")
            final_pass = False
            continue

        files = [
            f for f in os.listdir(folder)
            if f.lower().endswith(EXTENSIONS)
        ]

        valid = 0
        corrupt = 0
        hashes = set()
        duplicates = 0

        sizes = set()
        modes = set()

        for file in files:

            path = os.path.join(folder, file)

            try:

                with Image.open(path) as img:

                    img.verify()

                # Re-open after verify
                with Image.open(path) as img:

                    sizes.add(img.size)
                    modes.add(img.mode)

                with open(path, "rb") as f:

                    h = hashlib.md5(
                        f.read()
                    ).hexdigest()

                if h in hashes:
                    duplicates += 1
                else:
                    hashes.add(h)

                valid += 1

            except Exception:

                corrupt += 1

        total_files += len(files)

        print(
            f"{source:20s} | "
            f"{cls:15s} | "
            f"Files: {len(files):3d} | "
            f"Valid: {valid:3d} | "
            f"Corrupt: {corrupt:2d} | "
            f"Duplicate: {duplicates:2d} | "
            f"Size: {sizes} | "
            f"Mode: {modes}"
        )

        # ------------------------------------------------
        # ACCEPTANCE
        # ------------------------------------------------

        if valid != 300:
            final_pass = False

        if corrupt != 0:
            final_pass = False

        # Only accepted exception:
        # Internal Healthy may contain exactly 1 duplicate
        if source == "Internal_PlantVillage" and cls == "Healthy":

            if duplicates > 1:
                final_pass = False

        else:

            if duplicates != 0:
                final_pass = False


# ================================================================
# FINAL STATUS
# ================================================================

print("\n" + "=" * 72)
print("🍅 FINAL PREPROCESSING ACCEPTANCE")
print("=" * 72)

if final_pass:

    print("""
🎉 TOMATO PREPROCESSING — PASS

INTERNAL
Healthy        : 300 files
Early_Blight   : 300 files
Late_Blight    : 300 files

EXTERNAL
Healthy        : 300 files
Early_Blight   : 300 files
Late_Blight    : 300 files

Quality:
✅ All files readable
✅ No corrupt images
✅ Correct class folders
✅ 300 files/class
✅ Preprocessed data available
⚠️ Internal Healthy: 1 accepted duplicate

The duplicate is the previously documented
PlantVillage duplicate and is NOT being re-downloaded.

TOTAL PREPROCESSED FILES = 1,800
TOTAL UNIQUE CONTENT     = 1,799

================================================
🔒 TOMATO PREPROCESSED DATA LOCKED
================================================

NEXT:
➡️ TRAIN / VALIDATION / TEST SPLIT
➡️ MODEL TRAINING
""")

else:

    print("""
❌ PREPROCESSING LOCK FAILED

Do NOT start model training.

Check the class-wise results above.
""")

print("=" * 72)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🍅 TOMATO — FINAL PREPROCESSING LOCK

READ-ONLY CHECK

✅ No raw data modification
✅ No deletion
✅ No movement
✅ No resizing
✅ No reprocessing


INTERNAL_PLANTVILLAGE
Internal_PlantVillage | Healthy         | Files: 300 | Valid: 300 | Corrupt:  0 | Duplicate:  1 | Size: {(224, 224)} | Mode: {'RGB'}
Internal_PlantVillage | Early_Blight    | Files: 300 | Valid: 300 | Corrupt:  0 | Duplicate:  0 | Size: {(224, 224)} | Mode: {'RGB'}
Internal_PlantVillage | Late_Blight     | Files: 300 | Valid: 300 | Corrupt:  0 | Duplicate:  0 | Size: {(224, 224)} | Mode: {'RGB'}

EXTERNAL_NATURAL
External_Natural     | Healthy         | Files: 300 | Valid: 300 | Corrupt:  0 | Duplicate:  0 | Size: {(224, 224)} | Mode: {'RGB'}
External_Natural     | Early_Blight    | Files: 300 | Valid: 300 | Corrupt:  0 | Duplicate:  0 | Size: {(224, 224)} | Mode: {'RGB'}
External_Natural     | La

In [ ]:
# ================================================================
# 🍅 TOMATO — FINAL DATA CLEANING + QC
# INTERNAL + EXTERNAL
# READ-ONLY FINAL CHECK
# ================================================================

import os
import hashlib
import random
import csv
from PIL import Image
from google.colab import drive

drive.mount("/content/drive")

# ================================================================
# PATHS
# ================================================================

PROJECT = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

PREP = os.path.join(
    PROJECT,
    "2nd_Preprocessed_Data",
    "Tomato"
)

REPORT_DIR = os.path.join(
    PREP,
    "Final_QC"
)

os.makedirs(REPORT_DIR, exist_ok=True)

SOURCES = [
    "Internal_PlantVillage",
    "External_Natural"
]

CLASSES = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)

EXPECTED_FILES = 300
EXPECTED_SIZE = (224, 224)
EXPECTED_MODE = "RGB"

# ================================================================
# HEADER
# ================================================================

print("=" * 72)
print("🍅 TOMATO — FINAL DATA CLEANING + QUALITY CONTROL")
print("=" * 72)

print("""
READ-ONLY FINAL CHECK

✅ Count
✅ Corrupt image detection
✅ Duplicate detection
✅ Image size
✅ RGB check
✅ Class balance
✅ Internal / External structure

❌ No deletion
❌ No movement
❌ No renaming
❌ No modification
""")

# ================================================================
# SCAN
# ================================================================

results = []

overall_pass = True

for source in SOURCES:

    print("\n" + "=" * 72)
    print(source.upper())
    print("=" * 72)

    for cls in CLASSES:

        folder = os.path.join(
            PREP,
            source,
            cls
        )

        print(f"\n{source} → {cls}")

        if not os.path.exists(folder):

            print("❌ Folder missing")

            overall_pass = False
            continue

        files = sorted([
            f for f in os.listdir(folder)
            if f.lower().endswith(EXTENSIONS)
        ])

        total = len(files)
        valid = 0
        corrupt = 0
        duplicates = 0

        hashes = set()

        sizes = {}
        modes = {}

        for file in files:

            path = os.path.join(
                folder,
                file
            )

            try:

                # ------------------------------------------------
                # IMAGE CHECK
                # ------------------------------------------------

                with Image.open(path) as img:

                    img.load()

                    size = img.size
                    mode = img.mode

                # ------------------------------------------------
                # HASH
                # ------------------------------------------------

                with open(path, "rb") as f:

                    file_hash = hashlib.md5(
                        f.read()
                    ).hexdigest()

                if file_hash in hashes:

                    duplicates += 1

                else:

                    hashes.add(file_hash)

                # ------------------------------------------------
                # STATISTICS
                # ------------------------------------------------

                sizes[size] = sizes.get(size, 0) + 1
                modes[mode] = modes.get(mode, 0) + 1

                valid += 1

            except Exception:

                corrupt += 1

        unique = len(hashes)

        print(
            f"Files      : {total}"
        )

        print(
            f"Valid      : {valid}"
        )

        print(
            f"Corrupt    : {corrupt}"
        )

        print(
            f"Duplicate  : {duplicates}"
        )

        print(
            f"Unique     : {unique}"
        )

        print(
            f"Sizes      : {sizes}"
        )

        print(
            f"Modes      : {modes}"
        )

        # ========================================================
        # ACCEPTANCE RULE
        # ========================================================

        passed = True

        if total != EXPECTED_FILES:
            passed = False

        if valid != EXPECTED_FILES:
            passed = False

        if corrupt != 0:
            passed = False

        if sizes != {EXPECTED_SIZE: EXPECTED_FILES}:
            passed = False

        if modes != {EXPECTED_MODE: EXPECTED_FILES}:
            passed = False

        # --------------------------------------------------------
        # ACCEPTED EXCEPTION
        # --------------------------------------------------------

        if source == "Internal_PlantVillage" and cls == "Healthy":

            # One previously documented duplicate is accepted.
            if duplicates > 1:
                passed = False

        else:

            if duplicates != 0:
                passed = False

        status = "PASS" if passed else "FAIL"

        print(
            f"STATUS     : {status}"
        )

        if not passed:
            overall_pass = False

        results.append([
            source,
            cls,
            total,
            valid,
            corrupt,
            duplicates,
            unique,
            str(sizes),
            str(modes),
            status
        ])

# ================================================================
# SAVE REPORT
# ================================================================

report_path = os.path.join(
    REPORT_DIR,
    "Tomato_Final_Data_Cleaning_QC.csv"
)

with open(
    report_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "Source",
        "Class",
        "Total",
        "Valid",
        "Corrupt",
        "Duplicate",
        "Unique",
        "Dimensions",
        "Mode",
        "Status"
    ])

    writer.writerows(results)

print("\n" + "=" * 72)
print("📄 FINAL QC REPORT")
print("=" * 72)

print(report_path)

# ================================================================
# FINAL RESULT
# ================================================================

print("\n" + "=" * 72)
print("🏁 TOMATO FINAL DATA CLEANING RESULT")
print("=" * 72)

if overall_pass:

    print("""
🎉 FINAL CLEANING + QC PASSED

Internal:
    Healthy        = 300 files
    Early_Blight   = 300 files
    Late_Blight    = 300 files

External:
    Healthy        = 300 files
    Early_Blight   = 300 files
    Late_Blight    = 300 files

TOTAL FILES = 1,800

Quality:
✅ All images readable
✅ 224 × 224
✅ RGB
✅ No corrupt images
✅ Balanced classes
✅ Correct folder structure
⚠️ One previously documented Internal Healthy duplicate accepted

================================================
🔒 TOMATO DATA READY FOR SPLITTING
================================================

NEXT:
➡️ Train / Validation / Test Split
""")

else:

    print("""
❌ FINAL QC FAILED

Do NOT split or train yet.

Check the class-wise results above.
""")

print("=" * 72)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🍅 TOMATO — FINAL DATA CLEANING + QUALITY CONTROL

READ-ONLY FINAL CHECK

✅ Count
✅ Corrupt image detection
✅ Duplicate detection
✅ Image size
✅ RGB check
✅ Class balance
✅ Internal / External structure

❌ No deletion
❌ No movement
❌ No renaming
❌ No modification


INTERNAL_PLANTVILLAGE

Internal_PlantVillage → Healthy
Files      : 300
Valid      : 300
Corrupt    : 0
Duplicate  : 1
Unique     : 299
Sizes      : {(224, 224): 300}
Modes      : {'RGB': 300}
STATUS     : PASS

Internal_PlantVillage → Early_Blight
Files      : 300
Valid      : 300
Corrupt    : 0
Duplicate  : 0
Unique     : 300
Sizes      : {(224, 224): 300}
Modes      : {'RGB': 300}
STATUS     : PASS

Internal_PlantVillage → Late_Blight
Files      : 300
Valid      : 300
Corrupt    : 0
Duplicate  : 0
Unique     : 300
Sizes      : {(224, 224): 300}
Modes      : {'RGB': 300}
STATUS     : PASS

EXTERNA

In [ ]:
# ================================================================
# PLANT DISEASE PROJECT — COMPLETE DRIVE FOLDER AUDIT
# READ ONLY
# ================================================================

import os
from collections import Counter
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

IMAGE_EXT = (".jpg", ".jpeg", ".png", ".webp")

print("=" * 80)
print("PLANT DISEASE DETECTION — COMPLETE PROJECT DRIVE AUDIT")
print("=" * 80)

if not os.path.exists(PROJECT):
    raise FileNotFoundError("Project folder not found.")

total_files = 0
total_images = 0

# Walk through every folder
for root, dirs, files in os.walk(PROJECT):

    level = root.replace(PROJECT, "").count(os.sep)
    indent = "    " * level
    folder_name = os.path.basename(root)

    print(f"\n{indent}📁 {folder_name}")

    if files:

        image_files = [
            f for f in files
            if f.lower().endswith(IMAGE_EXT)
        ]

        other_files = [
            f for f in files
            if not f.lower().endswith(IMAGE_EXT)
        ]

        total_files += len(files)
        total_images += len(image_files)

        print(f"{indent}   Total files : {len(files)}")
        print(f"{indent}   Images      : {len(image_files)}")
        print(f"{indent}   Other files : {len(other_files)}")

        # Image formats
        if image_files:

            formats = Counter(
                os.path.splitext(f)[1].upper()
                for f in image_files
            )

            print(f"{indent}   Formats     : {dict(formats)}")

        # Show sample files
        print(f"{indent}   Sample files:")

        for f in files[:5]:
            print(f"{indent}      → {f}")

print("\n" + "=" * 80)
print("PROJECT SUMMARY")
print("=" * 80)

print("Project path :", PROJECT)
print("Total files  :", total_files)
print("Total images :", total_images)

print("\nImportant folders expected:")
print("1st Raw_Data")
print("2nd_Preprocessed_Data")
print("3rd Preprocessing")
print("5th Model_Evaluation")
print("6th Trained_Model")
print("7th Test_Images")
print("8th Web_Application")
print("9th Documentation")
print("10th Presentations (PPT)")

print("\nThis audit is READ-ONLY.")
print("No file was deleted, moved, renamed, or modified.")
print("=" * 80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PLANT DISEASE DETECTION — COMPLETE PROJECT DRIVE AUDIT

📁 Plant Disease Detection (Computer Vision)

    📁 2nd Data_Verification
       Total files : 3
       Images      : 0
       Other files : 3
       Sample files:
          → potato varification.ipynb
          → (OLD)lab_potatod_ataset_check.ipynb
          → tomato dataCollection+varification.ipynb

    📁 3rd Preprocessing
       Total files : 5
       Images      : 0
       Other files : 5
       Sample files:
          → Tomato_Preprocessing.ipynb
          → lab_potatod_preprocessing.ipynb
          → lab_potatod_processed_potato_data.npz
          → potato_processed_data.npz
          → Potato_Preprocessing.ipynb

    📁 4th Model_Training
       Total files : 2
       Images      : 0
       Other files : 2
       Sample files:
          → lab_potatod_CNN_training.ipynb
          → all potatod_CNN_t

In [6]:
# ================================================================
# 🍅 TOMATO — CREATE FINAL PROCESSED NPZ DATASET
# ================================================================
# Purpose:
# Package the already-cleaned and preprocessed Tomato images
# into the project's official 3rd Preprocessing folder.
#
# RAW DATA:
# 1st Raw_Data  → NEVER modified
#
# CLEAN/PREPROCESSED IMAGES:
# 2nd_Preprocessed_Data/Tomato
#
# FINAL PACKAGED DATA:
# 3rd Preprocessing/tomato_processed_data.npz
# ================================================================

import os
import numpy as np
from PIL import Image
from collections import Counter

# ================================================================
# 1. PROJECT PATHS
# ================================================================

BASE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

PREPROCESSED_ROOT = os.path.join(
    BASE,
    "2nd_Preprocessed_Data",
    "Tomato"
)

FINAL_ROOT = os.path.join(
    BASE,
    "3rd Preprocessing"
)

os.makedirs(FINAL_ROOT, exist_ok=True)

OUTPUT_NPZ = os.path.join(
    FINAL_ROOT,
    "tomato_processed_data.npz"
)

# ================================================================
# 2. DATA SOURCES
# ================================================================

SOURCES = {
    "Internal_PlantVillage": os.path.join(
        PREPROCESSED_ROOT,
        "Internal_PlantVillage"
    ),

    "External_Natural": os.path.join(
        PREPROCESSED_ROOT,
        "External_Natural"
    )
}

CLASS_NAMES = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

CLASS_TO_LABEL = {
    "Healthy": 0,
    "Early_Blight": 1,
    "Late_Blight": 2
}

# ================================================================
# 3. CHECK INPUT DATA
# ================================================================

print("=" * 75)
print("🍅 TOMATO — CREATING FINAL PROCESSED DATASET")
print("=" * 75)

for source_name, source_path in SOURCES.items():

    if not os.path.exists(source_path):
        raise FileNotFoundError(
            f"❌ Missing source folder:\n{source_path}"
        )

    print(f"✅ {source_name} found")

# ================================================================
# 4. LOAD IMAGES
# ================================================================

X = []
y = []
sources = []

summary = []

print("\n" + "=" * 75)
print("📥 LOADING PREPROCESSED TOMATO IMAGES")
print("=" * 75)

for source_name, source_path in SOURCES.items():

    print(f"\nSOURCE: {source_name}")

    for class_name in CLASS_NAMES:

        class_path = os.path.join(
            source_path,
            class_name
        )

        if not os.path.exists(class_path):
            raise FileNotFoundError(
                f"❌ Missing class folder:\n{class_path}"
            )

        files = sorted([
            f for f in os.listdir(class_path)
            if f.lower().endswith(
                (".png", ".jpg", ".jpeg", ".bmp", ".webp")
            )
        ])

        loaded = 0
        failed = 0

        for filename in files:

            filepath = os.path.join(
                class_path,
                filename
            )

            try:

                # Open image
                img = Image.open(filepath)

                # Ensure RGB
                img = img.convert("RGB")

                # Convert to NumPy array
                arr = np.array(img, dtype=np.uint8)

                # ------------------------------------------------
                # FINAL IMAGE SHAPE CHECK
                # ------------------------------------------------

                if arr.shape != (224, 224, 3):

                    print(
                        f"⚠️ Unexpected shape: "
                        f"{filename} → {arr.shape}"
                    )

                    failed += 1
                    continue

                X.append(arr)

                y.append(CLASS_TO_LABEL[class_name])

                sources.append(source_name)

                loaded += 1

            except Exception as e:

                print(
                    f"❌ Failed: {filename} | {e}"
                )

                failed += 1

        summary.append({
            "Source": source_name,
            "Class": class_name,
            "Files_Found": len(files),
            "Loaded": loaded,
            "Failed": failed
        })

        print(
            f"{class_name:15} | "
            f"Found: {len(files):3} | "
            f"Loaded: {loaded:3} | "
            f"Failed: {failed:2}"
        )

# ================================================================
# 5. CONVERT TO NUMPY ARRAYS
# ================================================================

X = np.array(X, dtype=np.uint8)
y = np.array(y, dtype=np.int64)
sources = np.array(sources)

class_names_array = np.array(
    CLASS_NAMES
)

# ================================================================
# 6. FINAL DATASET CHECK
# ================================================================

print("\n" + "=" * 75)
print("🔍 FINAL NPZ DATASET CHECK")
print("=" * 75)

print(f"X shape       : {X.shape}")
print(f"y shape       : {y.shape}")
print(f"sources shape : {sources.shape}")
print(f"Data type     : {X.dtype}")

print("\nClass distribution:")

for label, class_name in enumerate(CLASS_NAMES):

    count = np.sum(y == label)

    print(
        f"{class_name:15} : {count}"
    )

print("\nSource distribution:")

for source_name in SOURCES.keys():

    count = np.sum(sources == source_name)

    print(
        f"{source_name:25} : {count}"
    )

# ================================================================
# 7. VALIDATE TOTAL
# ================================================================

EXPECTED_TOTAL = 1800

if len(X) != EXPECTED_TOTAL:

    raise ValueError(
        f"❌ Expected {EXPECTED_TOTAL} images "
        f"but loaded {len(X)}"
    )

if len(y) != EXPECTED_TOTAL:

    raise ValueError(
        f"❌ Label count mismatch: {len(y)}"
    )

if len(sources) != EXPECTED_TOTAL:

    raise ValueError(
        f"❌ Source count mismatch: {len(sources)}"
    )

if X.shape[1:] != (224, 224, 3):

    raise ValueError(
        f"❌ Incorrect image shape: {X.shape}"
    )

# ================================================================
# 8. SAVE FINAL NPZ
# ================================================================

print("\n" + "=" * 75)
print("💾 SAVING FINAL TOMATO DATASET")
print("=" * 75)

np.savez_compressed(
    OUTPUT_NPZ,
    X=X,
    y=y,
    class_names=class_names_array,
    source=sources
)

print(f"✅ Saved successfully:")
print(OUTPUT_NPZ)

# ================================================================
# 9. CHECK FILE SIZE
# ================================================================

file_size_mb = os.path.getsize(
    OUTPUT_NPZ
) / (1024 ** 2)

print(
    f"\nNPZ file size: {file_size_mb:.2f} MB"
)

# ================================================================
# 10. RELOAD TEST
# ================================================================

print("\n" + "=" * 75)
print("🔄 RELOAD TEST")
print("=" * 75)

test_data = np.load(
    OUTPUT_NPZ,
    allow_pickle=False
)

X_test = test_data["X"]
y_test = test_data["y"]
class_names_test = test_data["class_names"]
sources_test = test_data["source"]

print(f"X             : {X_test.shape}")
print(f"y             : {y_test.shape}")
print(f"class_names   : {class_names_test}")
print(f"sources       : {sources_test.shape}")

# ================================================================
# 11. FINAL ACCEPTANCE
# ================================================================

if (
    X_test.shape == (1800, 224, 224, 3)
    and y_test.shape == (1800,)
    and sources_test.shape == (1800,)
    and len(class_names_test) == 3
):

    print("\n" + "=" * 75)
    print("🎉 TOMATO PROCESSED DATASET — FINAL PASS")
    print("=" * 75)

    print("✅ 1,800 images loaded")
    print("✅ 224 × 224 × 3")
    print("✅ RGB")
    print("✅ 3 classes")
    print("✅ Labels stored")
    print("✅ Source information stored")
    print("✅ NPZ saved")
    print("✅ NPZ reload successful")

    print("\nClasses:")
    print("0 → Healthy")
    print("1 → Early_Blight")
    print("2 → Late_Blight")

    print("\nSources:")
    print("Internal_PlantVillage → 900")
    print("External_Natural     → 900")

    print("\n" + "=" * 75)
    print("🔒 TOMATO PROCESSED DATASET LOCKED")
    print("=" * 75)

    print("\nFINAL FILE:")
    print(OUTPUT_NPZ)

    print("\nNEXT:")
    print("➡️ Train / Validation / Test Split")
    print("➡️ Model Training")

else:

    print("\n❌ FINAL NPZ VERIFICATION FAILED")
    print("Do NOT proceed to model training.")

🍅 TOMATO — CREATING FINAL PROCESSED DATASET
✅ Internal_PlantVillage found
✅ External_Natural found

📥 LOADING PREPROCESSED TOMATO IMAGES

SOURCE: Internal_PlantVillage
Healthy         | Found: 300 | Loaded: 300 | Failed:  0
Early_Blight    | Found: 300 | Loaded: 300 | Failed:  0
Late_Blight     | Found: 300 | Loaded: 300 | Failed:  0

SOURCE: External_Natural
Healthy         | Found: 300 | Loaded: 300 | Failed:  0
Early_Blight    | Found: 300 | Loaded: 300 | Failed:  0
Late_Blight     | Found: 300 | Loaded: 300 | Failed:  0

🔍 FINAL NPZ DATASET CHECK
X shape       : (1800, 224, 224, 3)
y shape       : (1800,)
sources shape : (1800,)
Data type     : uint8

Class distribution:
Healthy         : 600
Early_Blight    : 600
Late_Blight     : 600

Source distribution:
Internal_PlantVillage     : 900
External_Natural          : 900

💾 SAVING FINAL TOMATO DATASET
✅ Saved successfully:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/tomato_processed_data.npz

N